We'll set everything up here before writing to a .py

In [36]:
# importing our base model
import torch
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor
import torchvision


In [37]:
# I suspect this cell to somehow crash everytime
base_model = Qwen3VLForConditionalGeneration.from_pretrained("Qwen/Qwen3-VL-2B-Instruct", dtype=torch.float32, device_map="cpu")
base_processor = AutoProcessor.from_pretrained("Qwen/Qwen3-VL-2B-Instruct") 
tokenizer = base_processor.tokenizer

Loading weights: 100%|██████████| 625/625 [00:08<00:00, 71.42it/s] 


In [38]:
base_model = base_model.to("mps")

In [39]:
# Ok let's now start working on Phase I which uses the : https://huggingface.co/datasets/CraftJarvis/minecraft-vlp dataset
from huggingface_hub import snapshot_download, hf_hub_download
import huggingface_hub

# here for phase I we just need a single file from the repo
data_step_one = hf_hub_download(repo_id="CraftJarvis/minecraft-vlp", filename="mc-qa-250312.jsonl", repo_type="dataset", local_dir="./data_phase_one")


# we want to train to predict : P(assistant | system, user)

In [40]:
# for reference here is what one line of that json looks like :

'''
{"id": "qa-f89343ef-d687-49b5-95d5-1d0e9d63da18", "label": ["qa", "wiki", "self-instruct"], "image": [], 
"conversations": 
[{"role": "system", "content": [{"type": "text", "text": "You are a helpful assistant that can answer questions and help with tasks in Minecraft.\n"}]}, 
{"role": "user", "content": [{"type": "text", "text": "How do you enchant items in Minecraft, and what are some of the enchantments that can be applied?"}]}, 
{"role": "assistant", "content": [{"type": "text", "text": "You can enchant items in Minecraft by using an enchanting table, anvil, or grindstone. Enchantments that can be applied include but are not limited to: Sharpness, Protection, Efficiency, Unbreaking, Silk Touch, and Fortune."}]}]}


No image context for phase 1 ofc
'''

#Let's real quickly try to parse that line fine before doing it on the whole big ass json file
import json 

with open(data_step_one) as f:
    lines = f.readlines()

line = lines[12]
#print(line)

dict = json.loads(line)
dict["conversations"][1]["content"][0]['text'] # How do you enchant items in Minecraft, and what are some of the enchantments that can be applied? 
# the question
system=dict["conversations"][0]["content"][0]['text']
user = dict["conversations"][1]["content"][0]['text']
assistant = dict["conversations"][2]["content"][0]['text']

# format we want (with the Qwen tokens/markers and the format it was pretrained on)

pre_tokenization_datapoint = '<|im_start|>system\n'+system+'<|im_end|>\n'+'<|im_start|>user\n'+user+'<|im_end|>\n'+'<|im_start|>assistant\n'+assistant+'<|im_end|>\n'
print(pre_tokenization_datapoint)

'''
Good :

<|im_start|>system
You are a helpful assistant that can answer questions and help with tasks in Minecraft.
<|im_end|>
<|im_start|>user
How do you enchant items in Minecraft, and what are some of the enchantments that can be applied?<|im_end|>
<|im_start|>assistant
You can enchant items in Minecraft by using an enchanting table, anvil, or grindstone. Enchantments that can be applied include but are not limited to: Sharpness, Protection, Efficiency, Unbreaking, Silk Touch, and Fortune.<|im_end|>
'''



<|im_start|>system
You are a helpful assistant that can answer questions and help with tasks in Minecraft.
<|im_end|>
<|im_start|>user
How do you enchant items in Minecraft, and what are some of the enchantments that can be applied?<|im_end|>
<|im_start|>assistant
You can enchant items in Minecraft by using an enchanting table, anvil, or grindstone. Enchantments that can be applied include but are not limited to: Sharpness, Protection, Efficiency, Unbreaking, Silk Touch, and Fortune.<|im_end|>



'\nGood :\n\n<|im_start|>system\nYou are a helpful assistant that can answer questions and help with tasks in Minecraft.\n<|im_end|>\n<|im_start|>user\nHow do you enchant items in Minecraft, and what are some of the enchantments that can be applied?<|im_end|>\n<|im_start|>assistant\nYou can enchant items in Minecraft by using an enchanting table, anvil, or grindstone. Enchantments that can be applied include but are not limited to: Sharpness, Protection, Efficiency, Unbreaking, Silk Touch, and Fortune.<|im_end|>\n'

In [41]:
# now that we got the format right, let's do it over the whole json, and store it

with open('./data_phase_one/phase_one_formatted.jsonl', 'x') as f: # exclusive for no re-write unless manual deletion of file
    for line in lines:
        dict = json.loads(line)
        system=dict["conversations"][0]["content"][0]['text']
        user = dict["conversations"][1]["content"][0]['text']
        assistant = dict["conversations"][2]["content"][0]['text']
        text = '<|im_start|>system\n'+system+'<|im_end|>\n'+'<|im_start|>user\n'+user+'<|im_end|>\n'+'<|im_start|>assistant\n'+assistant+'<|im_end|>\n'
        f.write(json.dumps({'datapoint':text})+'\n')


FileExistsError: [Errno 17] File exists: './data_phase_one/phase_one_formatted.jsonl'

In [46]:
# now let's set up our proper dataset
# we want to keep the sentences separate, we hence want sumn like an array of arrays of tokens

with open('./data_phase_one/phase_one_formatted.jsonl', 'r') as f:
    buckets = [json.loads(line)['datapoint'] for line in f.readlines()]

#print(buckets[2]) we get what we want

token_buckets = [tokenizer(point, add_special_tokens=False) for point in buckets]

print(token_buckets[3]) # good, we get the token ids

{'input_ids': [151644, 8948, 198, 2610, 525, 264, 10950, 17847, 429, 646, 4226, 4755, 323, 1492, 448, 9079, 304, 40230, 624, 151645, 198, 151644, 872, 198, 4340, 653, 498, 1855, 264, 3731, 10812, 16224, 304, 40230, 323, 1128, 525, 1045, 315, 279, 2155, 5711, 369, 432, 30, 151645, 198, 151644, 77091, 198, 641, 40230, 11, 3731, 10812, 374, 264, 4911, 5101, 429, 646, 387, 1483, 311, 1855, 6351, 45021, 323, 7611, 13, 3731, 10812, 374, 1483, 311, 29282, 2355, 11, 20276, 23783, 11, 323, 68611, 5257, 9079, 304, 279, 1809, 13, 5692, 594, 264, 6770, 8474, 389, 1246, 311, 1855, 264, 3731, 10812, 16224, 323, 1045, 2155, 5711, 369, 432, 1447, 334, 16, 13, 48995, 3731, 10812, 25, 1019, 256, 481, 2014, 1191, 11, 498, 1184, 311, 10485, 3731, 10812, 16144, 10010, 11, 892, 525, 5990, 1730, 5538, 25260, 13, 1446, 3278, 1184, 458, 11001, 3735, 49739, 476, 2664, 311, 6530, 3731, 10812, 15797, 382, 334, 17, 13, 4230, 3731, 10812, 31993, 25, 1019, 256, 481, 9646, 498, 614, 3731, 10812, 16144, 11, 10485, 432

In [ ]:
# ok, now that we have that, we want to build our dataset, 